In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import us
from sklearn.preprocessing import StandardScaler

# Cleaning the NORS Outbreak Dataset

In [2]:
# Load the dataset
outbreak_df = pd.read_csv("NORS_outbreaks.csv", low_memory = False)

# View the data
outbreak_df.head()

,Year,Month,State,Primary Mode,Etiology,Serotype or Genotype,Etiology Status,Setting,Illnesses,Hospitalizations,Info On Hospitalizations,Deaths,Info On Deaths,Food Vehicle,Food Contaminated Ingredient,IFSAC Category,Water Exposure,Water Type,Animal Type
0,1971,2,California,Water,Copper,NaN,Confirmed,Restaurant,2,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Community,NaN
1,1971,6,Arkansas,Water,Hepatitis A,NaN,Confirmed,Store,98,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Other,NaN
2,1971,6,Missouri,Water,Unknown,NaN,Suspected,Subdivision/Neighborhood,2,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Community,NaN
3,1971,6,Alabama,Water,Selenium,NaN,Confirmed,Unknown,3,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Individual/Private,NaN
4,1971,6,Vermont,Water,Unknown,NaN,Suspected,Community/municipality,3,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Community,NaN


## Understand the Dataset

- Identify all columns (e.g., outbreak ID, state, illness type, number of cases, dates, settings).
- Check for categorical vs numeric variables.

In [3]:
# Show general info (types, missing values)
print("\nDataset Info:")
print(outbreak_df.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66713 entries, 0 to 66712
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Year                          66713 non-null  int64  
 1   Month                         66713 non-null  int64  
 2   State                         66713 non-null  object 
 3   Primary Mode                  66713 non-null  object 
 4   Etiology                      50375 non-null  object 
 5   Serotype or Genotype          16470 non-null  object 
 6   Etiology Status               50375 non-null  object 
 7   Setting                       60804 non-null  object 
 8   Illnesses                     66713 non-null  object 
 9   Hospitalizations              58155 non-null  float64
 10  Info On Hospitalizations      58480 non-null  object 
 11  Deaths                        58785 non-null  float64
 12  Info On Deaths                58463 non-null 

In [4]:
# Show summary statistics for numeric columns
print("\nSummary Statistics:")
display(outbreak_df.describe())


Summary Statistics:


,Year,Month,Hospitalizations,Deaths
count,66713.000000,66713.000000,58155.000000,58785.000000
mean,2012.696926,5.692009,0.825501,0.042375
std,7.546412,3.736764,4.164555,0.436484
min,1971.000000,1.000000,0.000000,0.000000
25%,2009.000000,2.000000,0.000000,0.000000
50%,2014.000000,5.000000,0.000000,0.000000
75%,2018.000000,9.000000,1.000000,0.000000
max,2023.000000,12.000000,308.000000,50.000000


In [5]:
# See unique values for key categorical columns
categorical_columns = ['State', 'Primary Mode']
for col in categorical_columns:
    if col in outbreak_df.columns:
        print(f"\nUnique values in {col}:")
        print(outbreak_df[col].unique())


Unique values in State:
['California' 'Arkansas' 'Missouri' 'Alabama' 'Vermont' 'Oregon'
 'New Jersey' 'Mississippi' 'Kentucky' 'Oklahoma' 'New Mexico'
 'North Carolina' 'New York' 'Alaska' 'Texas' 'Indiana' 'Colorado' 'Ohio'
 'Minnesota' 'Illinois' 'Florida' 'Pennsylvania' 'Washington' 'Maryland'
 'Massachusetts' 'Tennessee' 'Utah' 'Hawaii' 'Iowa' 'West Virginia'
 'Arizona' 'Virginia' 'Connecticut' 'Idaho' 'New Hampshire' 'Wisconsin'
 'Montana' 'Puerto Rico' 'Louisiana' 'Kansas' 'South Carolina' 'Maine'
 'Wyoming' 'North Dakota' 'Michigan' 'Georgia' 'South Dakota'
 'Rhode Island' 'Nevada' 'Virgin Islands' 'Multistate' 'Delaware'
 'Northern Mariana Islands' 'Nebraska' 'Guam' 'District of Columbia'
 'Republic of the Marshall Islands' 'Republic of Palau']

Unique values in Primary Mode:
['Water' 'Food' 'Person-to-person' 'Indeterminate/unknown'
 'Animal contact' 'Environmental contamination other than food/water']


## Handle Missing Data

- Remove columns with mostly missing values.
- Impute missing values if necessary (mean/mode or “Unknown”).

In [6]:
# Make a copy of the original dataset to clean
outbreak_clean = outbreak_df.copy()

In [7]:
# Drop the columns 'Info On Hospitalizations' and 'Info On Deaths'
columns_to_drop = ['Info On Hospitalizations', 'Info On Deaths']
for col in columns_to_drop:
    if col in outbreak_clean.columns:
        outbreak_clean = outbreak_clean.drop(columns=[col])

In [8]:
# Define categorical columns (general)
categorical_cols = ['Etiology', 'Etiology Status', 'Setting']

# Define contingent categorical columns
contingent_cols = [
    'Serotype or Genotype', 'Food Vehicle', 'Food Contaminated Ingredient',
    'IFSAC Category', 'Water Exposure', 'Water Type', 'Animal Type'
]

# Fill missing values in categorical columns with 'Unknown'
for col in categorical_cols:
    if col in outbreak_clean.columns:
        outbreak_clean[col] = outbreak_clean[col].fillna('Unknown')

# Fill missing values in contingent columns with 'Not Applicable'
for col in contingent_cols:
    if col in outbreak_clean.columns:
        outbreak_clean[col] = outbreak_clean[col].fillna('Not Applicable')

# Handle numeric columns
numeric_cols = ['Hospitalizations', 'Deaths']
for col in numeric_cols:
    if col in outbreak_clean.columns:
        outbreak_clean[col] = outbreak_clean[col].fillna(outbreak_clean[col].median())

# Check that all missing values are handled
print("Missing values after cleaning:")
print(outbreak_clean.isnull().sum())

Missing values after cleaning:
Year                            0
Month                           0
State                           0
Primary Mode                    0
Etiology                        0
Serotype or Genotype            0
Etiology Status                 0
Setting                         0
Illnesses                       0
Hospitalizations                0
Deaths                          0
Food Vehicle                    0
Food Contaminated Ingredient    0
IFSAC Category                  0
Water Exposure                  0
Water Type                      0
Animal Type                     0
dtype: int64


## Standardize and Normalize Data

- Standardize categorical variables (e.g., state abbreviations, illness names).
- Normalize numeric variables if you plan to use clustering or distance-based methods.

In [9]:
# Rename all columns to lowercase and replace spaces with underscores
outbreak_clean.columns = outbreak_clean.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('[^0-9a-zA-Z_]', '', regex=True)

# Create a 'date' column from 'year' and 'month'
outbreak_clean['date'] = pd.to_datetime(outbreak_clean[['year', 'month']].assign(day=1))

# Add a new column for state abbreviations
def get_state_abbr(state):
    state_info = us.states.lookup(state)
    return state_info.abbr if state_info else 'Unknown'

outbreak_clean['state_abbr'] = outbreak_clean['state'].apply(get_state_abbr)

In [16]:
# Standardize numeric values using z-score
# Select numeric columns to standardize
numeric_cols = ['hospitalizations', 'deaths']

# Initialize scaler
scaler = StandardScaler()

# Fit and transform numeric columns
outbreak_clean[numeric_cols] = scaler.fit_transform(outbreak_clean[numeric_cols])

# Preview the standardized numeric columns
display(outbreak_clean[numeric_cols].head())

,hospitalizations,deaths
0,-0.184607,-0.091081
1,-0.184607,-0.091081
2,-0.184607,-0.091081
3,-0.184607,-0.091081
4,-0.184607,-0.091081


## Feature Engineering
1. Outbreak Severity
Combine illnesses, hospitalizations, and deaths into a single severity measure. Since illnesses might still be an object (string), we first convert it to numeric if needed.

In [12]:
# Ensure illnesses column is numeric
outbreak_clean['illnesses'] = pd.to_numeric(outbreak_clean['illnesses'], errors='coerce').fillna(0)

# Create outbreak_severity as the sum of illnesses, hospitalizations, and deaths
outbreak_clean['outbreak_severity'] = outbreak_clean['illnesses'] + outbreak_clean['hospitalizations'] + outbreak_clean['deaths']

# Preview
display(outbreak_clean[['illnesses', 'hospitalizations', 'deaths', 'outbreak_severity']].head())

,illnesses,hospitalizations,deaths,outbreak_severity
0,2.0,-0.184607,-0.091081,1.724312
1,98.0,-0.184607,-0.091081,97.724312
2,2.0,-0.184607,-0.091081,1.724312
3,3.0,-0.184607,-0.091081,2.724312
4,3.0,-0.184607,-0.091081,2.724312


2. Binary Features Based on Primary Mode
Create flags for food, water, or animal-related outbreaks:

In [13]:
outbreak_clean['is_food_related'] = outbreak_clean['primary_mode'].apply(lambda x: 1 if 'Food' in x else 0)
outbreak_clean['is_water_related'] = outbreak_clean['primary_mode'].apply(lambda x: 1 if 'Water' in x else 0)
outbreak_clean['is_animal_related'] = outbreak_clean['primary_mode'].apply(lambda x: 1 if 'Animal' in x else 0)

# Preview
display(outbreak_clean[['primary_mode', 'is_food_related', 'is_water_related', 'is_animal_related']].head())

,primary_mode,is_food_related,is_water_related,is_animal_related
0,Water,0,1,0
1,Water,0,1,0
2,Water,0,1,0
3,Water,0,1,0
4,Water,0,1,0


3. Outbreak Month/Season
You can categorize outbreaks into seasons to see if there’s seasonality:

In [15]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

outbreak_clean['season'] = outbreak_clean['month'].apply(month_to_season)

# Preview
display(outbreak_clean[['month', 'season']].head())

,month,season
0,2,Winter
1,6,Summer
2,6,Summer
3,6,Summer
4,6,Summer


,date,year,month,state,state_abbr,primary_mode,etiology,ifsac_category,setting,illnesses,...,deaths,food_vehicle,water_exposure,water_type,animal_type,outbreak_severity,is_food_related,is_water_related,is_animal_related,season
0,1971-02-01,1971,2,California,CA,Water,Copper,Not Applicable,Restaurant,2.0,...,-0.091081,Not Applicable,Drinking water,Community,Not Applicable,1.724312,0,1,0,Winter
1,1971-06-01,1971,6,Arkansas,AR,Water,Hepatitis A,Not Applicable,Store,98.0,...,-0.091081,Not Applicable,Drinking water,Other,Not Applicable,97.724312,0,1,0,Summer
2,1971-06-01,1971,6,Missouri,MO,Water,Unknown,Not Applicable,Subdivision/Neighborhood,2.0,...,-0.091081,Not Applicable,Drinking water,Community,Not Applicable,1.724312,0,1,0,Summer
3,1971-06-01,1971,6,Alabama,AL,Water,Selenium,Not Applicable,Unknown,3.0,...,-0.091081,Not Applicable,Drinking water,Individual/Private,Not Applicable,2.724312,0,1,0,Summer
4,1971-06-01,1971,6,Vermont,VT,Water,Unknown,Not Applicable,Community/municipality,3.0,...,-0.091081,Not Applicable,Drinking water,Community,Not Applicable,2.724312,0,1,0,Summer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66708,2023-12-01,2023,12,Tennessee,TN,Person-to-person,Norovirus unknown,Not Applicable,Long-term care/nursing home/assisted living fa...,19.0,...,-0.091081,Not Applicable,Not Applicable,Not Applicable,Not Applicable,18.724312,0,0,0,Winter
66709,2023-12-01,2023,12,Tennessee,TN,Indeterminate/unknown,Norovirus Genogroup II,Not Applicable,Event space,18.0,...,-0.091081,Not Applicable,Not Applicable,Not Applicable,Not Applicable,17.724312,0,0,0,Winter
66710,2023-12-01,2023,12,Minnesota,MN,Person-to-person,Norovirus unknown,Not Applicable,School/college/university,7.0,...,-0.091081,Not Applicable,Not Applicable,Not Applicable,Not Applicable,6.724312,0,0,0,Winter
66711,2023-12-01,2023,12,Oregon,OR,Person-to-person,Unknown,Not Applicable,School/college/university,97.0,...,-0.091081,Not Applicable,Not Applicable,Not Applicable,Not Applicable,96.724312,0,0,0,Winter


In [20]:
# Reorder the columns
outbreak_clean = outbreak_clean[[col for col in [
    'date', 'year', 'month', 'season', 'state', 'state_abbr',
    'primary_mode', 'etiology', 'ifsac_category', 'setting',
    'illnesses', 'hospitalizations', 'deaths', 'outbreak_severity',
    'food_vehicle', 'food_contam_ingredient', 'water_exposure', 'water_type', 'animal_type',
    'is_food_related', 'is_water_related', 'is_animal_related'
] if col in outbreak_clean.columns]]

# Preview cleaned dataframe
display(outbreak_clean.head())

,date,year,month,season,state,state_abbr,primary_mode,etiology,ifsac_category,setting,...,hospitalizations,deaths,outbreak_severity,food_vehicle,water_exposure,water_type,animal_type,is_food_related,is_water_related,is_animal_related
0,1971-02-01,1971,2,Winter,California,CA,Water,Copper,Not Applicable,Restaurant,...,-0.184607,-0.091081,1.724312,Not Applicable,Drinking water,Community,Not Applicable,0,1,0
1,1971-06-01,1971,6,Summer,Arkansas,AR,Water,Hepatitis A,Not Applicable,Store,...,-0.184607,-0.091081,97.724312,Not Applicable,Drinking water,Other,Not Applicable,0,1,0
2,1971-06-01,1971,6,Summer,Missouri,MO,Water,Unknown,Not Applicable,Subdivision/Neighborhood,...,-0.184607,-0.091081,1.724312,Not Applicable,Drinking water,Community,Not Applicable,0,1,0
3,1971-06-01,1971,6,Summer,Alabama,AL,Water,Selenium,Not Applicable,Unknown,...,-0.184607,-0.091081,2.724312,Not Applicable,Drinking water,Individual/Private,Not Applicable,0,1,0
4,1971-06-01,1971,6,Summer,Vermont,VT,Water,Unknown,Not Applicable,Community/municipality,...,-0.184607,-0.091081,2.724312,Not Applicable,Drinking water,Community,Not Applicable,0,1,0
